<a href="https://colab.research.google.com/github/maruson08/new-folder-3/blob/main/8%EC%B0%A8%EC%8B%9C_DP_%EB%B0%9C%ED%91%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trials

In [ ]:
# =========================================================
# 🌌 Divide & Conquer Universe
# Maximum Subarray Space Simulator
# Colab Version
# =========================================================

# 실행 방법:
# 1. Colab에서 셀 전체 실행
# 2. 애니메이션 HTML 출력됨
# 3. Divide & Conquer / DP 비교 가능

# =========================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from matplotlib.patches import Circle
from IPython.display import HTML
from dataclasses import dataclass

# =========================================================
# 설정
# =========================================================

plt.style.use("dark_background")

# =========================================================
# 노드 클래스
# =========================================================

@dataclass
class SpaceNode:

    x: float
    y: float

    tx: float
    ty: float

    radius: float

    value: float

    depth: int

    alpha: float

    phase: str

    left: int
    right: int


# =========================================================
# 전역 상태
# =========================================================

frames = []

best_sum = float('-inf')
best_range = (0, 0)

# =========================================================
# 프레임 저장
# =========================================================

def save_frame(nodes, title):

    copied = []

    for n in nodes:

        copied.append(
            SpaceNode(
                n.x,
                n.y,
                n.tx,
                n.ty,
                n.radius,
                n.value,
                n.depth,
                n.alpha,
                n.phase,
                n.left,
                n.right
            )
        )

    frames.append((copied, title))


# =========================================================
# DP (Kadane)
# =========================================================

def dynamic_programming(arr):

    global best_sum
    global best_range

    current_sum = arr[0]
    max_sum = arr[0]

    start = end = temp = 0

    for i in range(1, len(arr)):

        if arr[i] > current_sum + arr[i]:
            current_sum = arr[i]
            temp = i
        else:
            current_sum += arr[i]

        if current_sum > max_sum:
            max_sum = current_sum
            start = temp
            end = i

    best_sum = max_sum
    best_range = (start, end)

    return max_sum


# =========================================================
# Divide & Conquer
# =========================================================

def max_crossing_sum(arr, left, mid, right):

    global best_sum
    global best_range

    left_sum = float('-inf')
    s = 0
    best_left = mid

    for i in range(mid, left - 1, -1):

        s += arr[i]

        if s > left_sum:
            left_sum = s
            best_left = i

    right_sum = float('-inf')
    s = 0
    best_right = mid + 1

    for i in range(mid + 1, right + 1):

        s += arr[i]

        if s > right_sum:
            right_sum = s
            best_right = i

    total = left_sum + right_sum

    if total > best_sum:
        best_sum = total
        best_range = (best_left, best_right)

    return total


# =========================================================
# 우주 시뮬레이션용 DnC
# =========================================================

def divide_and_conquer(
    arr,
    left,
    right,
    x,
    y,
    spread,
    depth,
    nodes
):

    size = right - left + 1

    value = np.sum(arr[left:right+1])

    node = SpaceNode(
        x=x,
        y=y,
        tx=x,
        ty=y,
        radius=max(0.25, size * 0.18),
        value=value,
        depth=depth,
        alpha=1.0,
        phase="DIVIDE",
        left=left,
        right=right
    )

    nodes.append(node)

    save_frame(nodes,
               f"DIVIDE [{left}, {right}]")

    if left == right:

        node.phase = "BASE"

        save_frame(nodes,
                   f"BASE [{left}]")

        return arr[left]

    mid = (left + right) // 2

    left_sum = divide_and_conquer(
        arr,
        left,
        mid,
        x - spread,
        y - 1.6,
        spread * 0.55,
        depth + 1,
        nodes
    )

    right_sum = divide_and_conquer(
        arr,
        mid + 1,
        right,
        x + spread,
        y - 1.6,
        spread * 0.55,
        depth + 1,
        nodes
    )

    cross_sum = max_crossing_sum(
        arr,
        left,
        mid,
        right
    )

    result = max(
        left_sum,
        right_sum,
        cross_sum
    )

    node.phase = "MERGE"

    node.value = result

    save_frame(nodes,
               f"MERGE [{left}, {right}]")

    return result


# =========================================================
# 애니메이션
# =========================================================

def animate_universe(arr, dnc_result, dp_result):

    fig, ax = plt.subplots(figsize=(15, 9))

    fig.patch.set_facecolor("#050816")

    def update(frame_idx):

        ax.clear()

        ax.set_facecolor("#050816")

        nodes, title = frames[frame_idx]

        # 별 배경
        np.random.seed(0)

        stars_x = np.random.uniform(-8, 8, 120)
        stars_y = np.random.uniform(-8, 2, 120)

        ax.scatter(
            stars_x,
            stars_y,
            s=np.random.uniform(1, 12, 120),
            alpha=0.45,
            color="white"
        )

        # 연결선
        for i in range(len(nodes)-1):

            n1 = nodes[i]

            for j in range(i+1, len(nodes)):

                n2 = nodes[j]

                dist = np.sqrt(
                    (n1.x - n2.x)**2 +
                    (n1.y - n2.y)**2
                )

                if dist < 3:

                    ax.plot(
                        [n1.x, n2.x],
                        [n1.y, n2.y],
                        alpha=0.08,
                        color="#00ffe1"
                    )

        # 노드
        for node in nodes:

            intensity = min(
                1.0,
                abs(node.value) / max(1, abs(best_sum))
            )

            # 색상
            if node.phase == "MERGE":
                color = (1.0, 0.3, intensity)

            elif node.phase == "BASE":
                color = (0.3, 0.7, 1.0)

            else:
                color = (0.0, intensity, 1.0)

            # 최적 구간 glow
            if (
                node.left == best_range[0]
                and
                node.right == best_range[1]
            ):

                glow = Circle(
                    (node.x, node.y),
                    node.radius * 2.2,
                    color="#00ffd5",
                    alpha=0.15
                )

                ax.add_patch(glow)

            # 본체
            circle = Circle(
                (node.x, node.y),
                node.radius,
                color=color,
                alpha=0.85
            )

            ax.add_patch(circle)

            ax.text(
                node.x,
                node.y,
                f"{int(node.value)}",
                color="white",
                fontsize=9,
                ha='center',
                va='center'
            )

        # HUD
        hud = (
            "🌌 DIVIDE & CONQUER UNIVERSE\n\n"
            f"FRAME        : {frame_idx+1}/{len(frames)}\n"
            f"CURRENT      : {title}\n"
            f"BEST SUM     : {best_sum}\n"
            f"BEST RANGE   : {best_range}\n\n"
            f"D&C RESULT   : {dnc_result}\n"
            f"DP RESULT    : {dp_result}\n\n"
            "ALGORITHMS\n"
            "- Divide & Conquer\n"
            "- Dynamic Programming\n"
        )

        ax.text(
            -7.7,
            1.6,
            hud,
            fontsize=11,
            color="#00ffe1",
            family="monospace",
            verticalalignment='top',
            bbox=dict(
                facecolor="#0b1020",
                edgecolor="#00ffe1",
                alpha=0.9
            )
        )

        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 2)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_title(
            "Recursive Space Simulator",
            fontsize=22,
            color="white",
            pad=20
        )

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=850,
        repeat=False
    )

    plt.close()

    return HTML(ani.to_jshtml())


# =========================================================
# 실행
# =========================================================

arr = [-2, 1, -3, 4, -1, 2, 1, -5, 4, 9, 30, -99]

# DP
dp_result = dynamic_programming(arr)

# D&C
nodes = []

dnc_result = divide_and_conquer(
    arr,
    0,
    len(arr)-1,
    x=0,
    y=0,
    spread=3.6,
    depth=0,
    nodes=nodes
)

print("=" * 60)
print("🌌 RECURSIVE SPACE SIMULATOR")
print("=" * 60)

print("Array:", arr)
print("Divide & Conquer :", dnc_result)
print("Dynamic Programming :", dp_result)

print("=" * 60)

# 애니메이션 출력
animate_universe(
    arr,
    dnc_result,
    dp_result
)

🌌 RECURSIVE SPACE SIMULATOR
Array: [-2, 1, -3, 4, -1, 2, 1, -5, 4, 9, 30, -99]
Divide & Conquer : 44
Dynamic Programming : 44


/tmp/ipykernel_3373/2859959356.py:423: UserWarning: Glyph 127756 (\N{MILKY WAY}) missing from font(s) DejaVu Sans Mono.
  return HTML(ani.to_jshtml())


In [ ]:
import time
import random
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# =========================================================
# Algorithms
# =========================================================

def dnc(arr, l, r):
    if l == r:
        return arr[l]

    m = (l + r) // 2

    left = dnc(arr, l, m)
    right = dnc(arr, m+1, r)

    s = 0
    left_best = float('-inf')
    for i in range(m, l-1, -1):
        s += arr[i]
        left_best = max(left_best, s)

    s = 0
    right_best = float('-inf')
    for i in range(m+1, r+1):
        s += arr[i]
        right_best = max(right_best, s)

    return max(left, right, left_best + right_best)


def kadane(arr):
    cur = best = arr[0]
    for x in arr[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


def prefix(arr):
    p = 0
    minp = 0
    best = float('-inf')

    for x in arr:
        p += x
        best = max(best, p - minp)
        minp = min(minp, p)

    return best


algorithms = {
    "DnC": lambda a: dnc(a, 0, len(a)-1),
    "Kadane": kadane,
    "Prefix": prefix
}

# =========================================================
# Data Generator
# =========================================================

def generate(n, mode):

    if mode == "random":
        return [random.randint(-100, 100) for _ in range(n)]

    if mode == "monotonic":
        return list(range(n))

    if mode == "alternating":
        return [(-1)**i * i for i in range(n)]

    if mode == "sparse":
        arr = [0]*n
        arr[n//2] = 1000
        return arr

# =========================================================
# MODE A: 알고리즘 1개 + 여러 테스트
# =========================================================

def mode_algorithm_focus(algo_name):

    sizes = list(range(100, 1000, 100))

    modes = ["random", "monotonic", "alternating", "sparse"]

    plt.figure(figsize=(9,6))

    for mode in modes:

        times = []

        for n in sizes:

            arr = generate(n, mode)

            t0 = time.perf_counter()
            algorithms[algo_name](arr)
            times.append((time.perf_counter() - t0) * 1000)

        plt.plot(
            sizes,
            times,
            marker='o',
            label=mode
        )

    plt.title(f"{algo_name} - Input Distribution Comparison")
    plt.xlabel("Input Size")
    plt.ylabel("Time (ms)")
    plt.grid()
    plt.legend()
    plt.show()
# =========================================================
# MODE B: 테스트 1개 + 여러 알고리즘
# =========================================================

def mode_case_focus(mode, size):

    arr = generate(size, mode)

    results = {}

    for name, func in algorithms.items():

        t0 = time.perf_counter()
        func(arr)
        results[name] = (time.perf_counter() - t0)*1000

    plt.figure(figsize=(8,5))
    plt.bar(results.keys(), results.values())
    plt.title(f"Algorithm comparison (mode={mode}, size={size})")
    plt.ylabel("Time (ms)")
    plt.grid(axis="y")
    plt.show()

# =========================================================
# UI
# =========================================================

mode_select = widgets.ToggleButtons(
    options=["Algorithm Focus", "Case Focus"]
)

algo_select = widgets.Dropdown(
    options=["DnC", "Kadane", "Prefix"],
    value="Kadane"
)

data_mode = widgets.Dropdown(
    options=["random", "monotonic", "alternating", "sparse"],
    value="random"
)

size_slider = widgets.IntSlider(
    value=300,
    min=50,
    max=1500,
    step=50
)

out = widgets.Output()

def update(change):

    with out:
        clear_output(wait=True)

        if mode_select.value == "Algorithm Focus":
            mode_algorithm_focus(
                algo_select.value
            )

        else:
            mode_case_focus(
                data_mode.value,
                size_slider.value
            )

display(mode_select, algo_select, data_mode, size_slider, out)

for w in [mode_select, algo_select, data_mode, size_slider]:
    w.observe(update, names="value")

update(None)

ToggleButtons(options=('Algorithm Focus', 'Case Focus'), value='Algorithm Focus')

Dropdown(index=1, options=('DnC', 'Kadane', 'Prefix'), value='Kadane')

Dropdown(options=('random', 'monotonic', 'alternating', 'sparse'), value='random')

IntSlider(value=300, max=1500, min=50, step=50)

Output()

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def divide_and_conquer(arr, left, right):
    """
    Divide & Conquer 최대 부분합 알고리즘

    핵심 아이디어:
    - 배열을 반으로 나눔 (Divide)
    - 각각 재귀적으로 해결 (Conquer)
    - 중간을 걸치는 구간 계산 (Combine)

    시간복잡도:
    T(n) = 2T(n/2) + O(n)
    → O(n log n)
    """

    # base case: 원소 1개면 자기 자신이 최대
    if left == right:
        return arr[left]

    mid = (left + right) // 2

    # 1. 왼쪽 구간 최대합
    left_max = divide_and_conquer(arr, left, mid)

    # 2. 오른쪽 구간 최대합
    right_max = divide_and_conquer(arr, mid + 1, right)

    # 3. 중간을 걸치는 경우 계산
    # (mid 기준 왼쪽으로 확장)
    s = 0
    left_sum = float('-inf')
    for i in range(mid, left - 1, -1):
        s += arr[i]
        left_sum = max(left_sum, s)

    # (mid+1 기준 오른쪽으로 확장)
    s = 0
    right_sum = float('-inf')
    for i in range(mid + 1, right + 1):
        s += arr[i]
        right_sum = max(right_sum, s)

    # 4. 3가지 중 최대 반환
    return max(left_max, right_max, left_sum + right_sum)

In [ ]:
def kadane(arr):
    """
    Dynamic Programming (Kadane Algorithm)

    핵심 아이디어:
    - 이전 상태(cur)를 기억
    - 현재 원소 포함 vs 새로 시작 비교

    상태 정의:
    cur[i] = i에서 끝나는 최대 부분합

    점화식:
    cur = max(x, cur + x)

    시간복잡도:
    O(n)
    """

    cur = best = arr[0]

    for x in arr[1:]:
        # 현재 원소를 포함하는 경우 vs 새로 시작
        cur = max(x, cur + x)

        # 전체 최댓값 갱신
        best = max(best, cur)

    return best

In [ ]:
def prefix_trick(arr):
    """
    Prefix Sum 기반 최적화 기법

    핵심 아이디어:
    - 누적합(prefix)을 이용
    - 구간합 = prefix[j] - prefix[i]

    최대 부분합 변환:
    max(prefix[j] - min(prefix[i]))

    시간복잡도:
    O(n)

    특징:
    - DP라기보다 '수학적 변환'
    - 상태 저장 없이 해결 가능
    """

    prefix = 0        # 현재까지 누적합
    min_prefix = 0    # 이전 prefix 중 최소값
    best = float('-inf')

    for x in arr:
        prefix += x

        # 현재 위치까지 최대 구간합
        best = max(best, prefix - min_prefix)

        # 최소 prefix 갱신
        min_prefix = min(min_prefix, prefix)

    return best

In [ ]:
def generate(n, mode="random"):

    if mode == "random":
        return [random.randint(-100, 100) for _ in range(n)]

    if mode == "monotonic":
        return list(range(n))

    if mode == "alternating":
        return [(-1)**i * i for i in range(n)]

    if mode == "sparse":
        arr = [0]*n
        arr[n//2] = 1000
        return arr

In [ ]:
def mode_A(algo_name, mode):

    sizes = list(range(100, 1200, 100))

    X = []
    y = []

    for n in sizes:
        arr = generate(n, mode)

        start = time.perf_counter()
        algos[algo_name](arr)
        end = time.perf_counter()

        X.append(n)
        y.append((end-start)*1000)

    X = np.array(X).reshape(-1,1)
    y = np.array(y)

    # ---------------------------
    # regression models
    # ---------------------------

    lin = LinearRegression()
    lin.fit(X, y)

    quad = make_pipeline(PolynomialFeatures(2), LinearRegression())
    quad.fit(X, y)

    # n log n model
    X_log = X * np.log2(X)
    log_model = LinearRegression()
    log_model.fit(X_log, y)

    # ---------------------------
    # PLOT (real data + models)
    # ---------------------------

    plt.figure(figsize=(10,6))

    # 🔴 REAL DATA (이게 핵심)
    plt.scatter(X, y, color="white", label="Real Data")

    # 📉 linear fit
    plt.plot(X, lin.predict(X), label="O(n) fit")

    # 📉 quadratic fit
    plt.plot(X, quad.predict(X), label="O(n^2) fit")

    # 📉 n log n fit
    plt.plot(X, log_model.predict(X_log), label="O(n log n) fit")

    plt.title(f"[MODE A] {algo_name} performance ({mode})")
    plt.xlabel("Input size")
    plt.ylabel("Time (ms)")
    plt.grid()
    plt.legend()
    plt.show()

In [ ]:
def mode_B(mode, size):

    arr = generate(size, mode)

    results = {}

    for name, f in algos.items():

        start = time.perf_counter()
        f(arr)
        results[name] = (time.perf_counter()-start)*1000

    plt.figure(figsize=(8,5))

    plt.bar(results.keys(), results.values())

    plt.title(f"[MODE B] Algorithm comparison ({mode}, n={size})")
    plt.ylabel("Time (ms)")
    plt.grid(axis="y")

    plt.show()

In [ ]:
mode_select = widgets.ToggleButtons(
    options=["Mode A (Algorithm Focus)", "Mode B (Case Focus)"]
)

algo_select = widgets.Dropdown(
    options=list(algos.keys()),
    value="Kadane"
)

data_mode = widgets.Dropdown(
    options=["random", "monotonic", "alternating", "sparse"],
    value="random"
)

size_slider = widgets.IntSlider(
    value=400,
    min=100,
    max=1500,
    step=100
)

out = widgets.Output()

In [ ]:
def update(change):

    with out:
        clear_output(wait=True)

        if mode_select.value == "Mode A (Algorithm Focus)":
            mode_A(algo_select.value, data_mode.value)

        else:
            mode_B(data_mode.value, size_slider.value)

In [ ]:
for w in [mode_select, algo_select, data_mode, size_slider]:
    w.observe(update, names="value")

display(mode_select, algo_select, data_mode, size_slider, out)

update(None)

ToggleButtons(options=('Mode A (Algorithm Focus)', 'Mode B (Case Focus)'), value='Mode A (Algorithm Focus)')

Dropdown(index=1, options=('DnC', 'Kadane', 'Prefix'), value='Kadane')

Dropdown(options=('random', 'monotonic', 'alternating', 'sparse'), value='random')

IntSlider(value=1500, max=1500, min=100, step=100)

Output()

# 3

## D&C

In [ ]:
# ── 방법 1: 분할정복 ──────────────────────────────────────────

def max_crossing_sum(arr, left, mid, right):
    left_sum = float('-inf')
    s = 0
    for i in range(mid, left - 1, -1):
        s += arr[i]
        left_sum = max(left_sum, s)

    right_sum = float('-inf')
    s = 0
    for i in range(mid + 1, right + 1):
        s += arr[i]
        right_sum = max(right_sum, s)

    # 두 합이 모두 유효할 때만 합산 (방어 코드)
    if left_sum == float('-inf') or right_sum == float('-inf'):
        return float('-inf')
    return left_sum + right_sum


def divide_and_conquer(arr, left, right):
    # 빈 배열 방어
    if not arr:
        raise ValueError("배열이 비어 있습니다.")

    if left == right:
        return arr[left]

    mid = (left + right) // 2
    left_max  = divide_and_conquer(arr, left, mid)
    right_max = divide_and_conquer(arr, mid + 1, right)
    cross_max = max_crossing_sum(arr, left, mid, right)

    return max(left_max, right_max, cross_max)


arr = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print(divide_and_conquer(arr, 0, len(arr) - 1))  # 6

6


In [ ]:
# 사용자 입력
user_input = input("배열을 입력하세요 (예: -2,1,-3,4,-1,2,1,-5,4): ")

try:
    arr = list(map(int, user_input.split(',')))
    if not arr:
        raise ValueError("배열이 비어 있습니다.")
except ValueError as e:
    print(f"입력 오류: {e}")
    exit()

print(divide_and_conquer(arr, 0, len(arr) - 1))  # 6

배열을 입력하세요 (예: -2,1,-3,4,-1,2,1,-5,4): -2, 1, -3, 4, -1, 2, 1, -5, 4
6


## DP

In [ ]:

# ── 방법 2: 동적계획법 (Kadane's Algorithm) ───────────────────

def dynamic_programming(arr):
    # 빈 배열 방어
    if not arr:
        raise ValueError("배열이 비어 있습니다.")

    max_sum     = arr[0]   # 전체 최댓값
    current_sum = arr[0]   # 현재까지의 서브배열 합

    for i in range(1, len(arr)):
        # 현재 원소만 취하는 게 나은지, 이어가는 게 나은지 선택
        current_sum = max(arr[i], current_sum + arr[i])
        max_sum     = max(max_sum, current_sum)

    return max_sum


arr = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print(dynamic_programming(arr))  # 6

6


In [ ]:
# 사용자 입력
user_input = input("배열을 입력하세요 (예: -2,1,-3,4,-1,2,1,-5,4): ")

try:
    arr = list(map(int, user_input.split(',')))
    if not arr:
        raise ValueError("배열이 비어 있습니다.")
except ValueError as e:
    print(f"입력 오류: {e}")
    exit()

print(dynamic_programming(arr))

배열을 입력하세요 (예: -2,1,-3,4,-1,2,1,-5,4): -2, 1, -3, 4, -1, 2, 1, -5, 4
6


## Test code

In [ ]:
import time

# ── 테스트 케이스 ─────────────────────────────────────────────
test_cases = [
    # (입력 배열, 기댓값, 설명, 카테고리)

    # 기본
    ([-2, 1, -3, 4, -1, 2, 1, -5, 4], 6,   "기본 예제",            "기본"),

    # 원소 개수 경계
    ([5],                               5,   "원소 1개 (양수)",       "경계"),
    ([-5],                             -5,   "원소 1개 (음수)",       "경계"),
    ([0],                               0,   "원소 1개 (0)",          "경계"),
    ([1, 2],                            3,   "원소 2개",              "경계"),

    # 부호
    ([-1, -2, -3, -4],                 -1,   "전부 음수",             "부호"),
    ([1, 2, 3, 4, 5],                  15,   "전부 양수",             "부호"),
    ([0, 0, 0, 0],                      0,   "전부 0",               "부호"),

    # 위치
    ([5, -9, -9, -9],                   5,   "최대 구간이 맨 앞",      "위치"),
    ([-9, -9, -9, 5],                   5,   "최대 구간이 맨 뒤",      "위치"),
    ([-3, 4, 6, -2, -9],               10,   "최대 구간이 중간",       "위치"),
    ([100, -1, -1, 100],              198,   "양 끝이 큰 경우",        "위치"),

    # 특수
    ([0, -1, 0, -1, 0],                 0,   "0과 음수 혼합",          "특수"),
    ([-1, 0, -1, 0, -1],               0,   "음수와 0 혼합",          "특수"),
    ([1, -1, 1, -1, 1],                1,   "양음 반복",              "특수"),
    (list(range(-50, 51)),           1275,   "범위 배열 (-50 ~ 50)",   "특수"),
    ([10**6, -1, 10**6],          1999999,   "큰 수 포함",             "특수"),
]

# ── 테스트 실행 ───────────────────────────────────────────────
def run_tests():
    categories = sorted(set(c for _, _, _, c in test_cases))

    total_dc_pass = 0
    total_dp_pass = 0

    for cat in categories:
        cases = [(a, e, d) for a, e, d, c in test_cases if c == cat]

        print(f"\n{'━' * 65}")
        print(f"  📂 [{cat}] 테스트 ({len(cases)}개)")
        print(f"{'━' * 65}")
        print(f"  {'#':<3} {'설명':<22} {'DC결과':<10} {'DP결과':<10} {'기댓값':<10} {'DC시간':>8} {'DP시간':>8}  판정")
        print(f"  {'-' * 62}")

        for i, (arr, expected, desc) in enumerate(cases, 1):
            # 분할정복 실행 + 시간 측정
            t0 = time.perf_counter()
            dc_result = divide_and_conquer(arr, 0, len(arr) - 1)
            dc_time = (time.perf_counter() - t0) * 1000  # ms

            # 동적계획법 실행 + 시간 측정
            t0 = time.perf_counter()
            dp_result = dynamic_programming(arr)
            dp_time = (time.perf_counter() - t0) * 1000  # ms

            dc_ok = dc_result == expected
            dp_ok = dp_result == expected
            if dc_ok: total_dc_pass += 1
            if dp_ok: total_dp_pass += 1

            판정 = "✅✅" if dc_ok and dp_ok else ("✅❌" if dc_ok else ("❌✅" if dp_ok else "❌❌"))

            print(f"  {i:<3} {desc:<22} {str(dc_result):<10} {str(dp_result):<10} {str(expected):<10} {dc_time:>6.3f}ms {dp_time:>6.3f}ms  {판정}")

    # ── 최종 요약 ─────────────────────────────────────────────
    total = len(test_cases)
    print(f"\n{'═' * 65}")
    print(f"  📊 최종 결과")
    print(f"{'═' * 65}")
    print(f"  분할정복  : {total_dc_pass}/{total} 통과 {'✅ ALL PASS' if total_dc_pass == total else '❌ 일부 실패'}")
    print(f"  동적계획법: {total_dp_pass}/{total} 통과 {'✅ ALL PASS' if total_dp_pass == total else '❌ 일부 실패'}")
    print(f"{'═' * 65}")

run_tests()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📂 [경계] 테스트 (4개)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  #   설명                     DC결과       DP결과       기댓값            DC시간     DP시간  판정
  --------------------------------------------------------------
  1   원소 1개 (양수)             5          5          5           0.002ms  0.003ms  ✅✅
  2   원소 1개 (음수)             -5         -5         -5          0.001ms  0.001ms  ✅✅
  3   원소 1개 (0)              0          0          0           0.001ms  0.000ms  ✅✅
  4   원소 2개                  3          3          3           0.010ms  0.001ms  ✅✅

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📂 [기본] 테스트 (1개)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  #   설명                     DC결과       DP결과       기댓값            DC시간     DP시간  판정
  --------------------------------------------------------------
  1   기본 예제                  6          6          6           0

# DnC vs DP vs Prefix

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import StandardScaler

In [ ]:
def dnc(arr, l, r):
    """
    Divide & Conquer Maximum Subarray Sum

    시간복잡도:
        T(n) = 2T(n/2) + O(n)
        → O(n log n)

    핵심:
        1. 왼쪽 최대
        2. 오른쪽 최대
        3. 중앙 crossing 최대
    """

    # base case: 원소 1개
    if l == r:
        return arr[l]

    mid = (l + r) // 2

    # 재귀적으로 좌/우 계산
    left = dnc(arr, l, mid)
    right = dnc(arr, mid+1, r)

    # --------------------------
    # crossing sum 계산
    # --------------------------

    # 왼쪽 확장
    s = 0
    left_best = float('-inf')
    for i in range(mid, l-1, -1):
        s += arr[i]
        left_best = max(left_best, s)

    # 오른쪽 확장
    s = 0
    right_best = float('-inf')
    for i in range(mid+1, r+1):
        s += arr[i]
        right_best = max(right_best, s)

    return max(left, right, left_best + right_best)

In [ ]:
def kadane(arr):
    """
    Dynamic Programming (Kadane Algorithm)

    상태 정의:
        cur[i] = i에서 끝나는 최대 부분합

    점화식:
        cur = max(x, cur + x)

    시간복잡도:
        O(n)
    """

    cur = best = arr[0]

    for x in arr[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)

    return best

In [ ]:
def prefix(arr):
    """
    Prefix Sum 기반 최적화 기법

    핵심 아이디어:
    - 누적합(prefix)을 이용
    - 구간합 = prefix[j] - prefix[i]

    최대 부분합 변환:
    max(prefix[j] - min(prefix[i]))

    시간복잡도:
    O(n)

    특징:
    - DP라기보다 '수학적 변환'
    - 상태 저장 없이 해결 가능
    """

    prefix = 0        # 현재까지 누적합
    min_prefix = 0    # 이전 prefix 중 최소값
    best = float('-inf')

    for x in arr:
        prefix += x

        # 현재 위치까지 최대 구간합
        best = max(best, prefix - min_prefix)

        # 최소 prefix 갱신
        min_prefix = min(min_prefix, prefix)

    return best

In [ ]:
algos = {
    "DnC": lambda a: dnc(a, 0, len(a)-1),
    "Kadane": kadane,
    "Prefix": prefix
}

In [ ]:
def generate(n, mode):
    """
    다양한 입력 분포 생성

    mode에 따라 알고리즘 성능이 달라짐
    (이게 이 프로젝트의 핵심 포인트)
    """

    if mode == "random":
        return [random.randint(-100, 100) for _ in range(n)]

    if mode == "monotonic":
        return list(range(n))

    if mode == "alternating":
        return [(-1)**i * i for i in range(n)]

    if mode == "sparse":
        arr = [0]*n
        arr[n//2] = 1000
        return arr

In [ ]:
def theory_curves(X):
    """
    📌 Big-O theoretical baseline curves

    - O(n)
    - O(n log n)
    - O(n^2)
    """

    X = X.reshape(-1)

    return {
        "O(n)": X,
        "O(n log n)": X * np.log2(X + 1),
        "O(n^2)": X ** 2
    }

In [ ]:
def mode_A(algo_name, mode, max_n):

    max_n = int(max_n)

    if max_n < 100:
        print("max_n must be >= 100")
        return

    # -----------------------------
    # STEP 자동 조정 (핵심)
    # -----------------------------
    step = max(100, max_n // 10)

    sizes = np.arange(100, max_n + 1, step)

    # -----------------------------
    # 강제 보정 (n=100일 때 최소 1개 보장)
    # -----------------------------
    if len(sizes) == 0:
        sizes = np.array([100])

    if sizes[-1] != max_n:
        sizes = np.append(sizes, max_n)

    real_times = []

    # --------------------------
    # warm-up (JIT-like effect 제거)
    # --------------------------
    algos[algo_name](generate(200, mode))

    # --------------------------
    # stable measurement
    # --------------------------
    def measure(func, arr, repeat=30):
        total = 0
        for _ in range(repeat):
            start = time.perf_counter()
            func(arr)
            end = time.perf_counter()
            total += (end - start)
        return (total / repeat) * 1000

    for n in sizes:
        arr = generate(n, mode)
        real_times.append(measure(algos[algo_name], arr))

    real_times = np.array(real_times)

    # --------------------------
    # theory curves
    # --------------------------
    theory = {
        "O(n)": sizes,
        "O(n log n)": sizes * np.log2(sizes + 1)
    }

    # --------------------------
    # ML (stable regression)
    # --------------------------
    X_feat = np.column_stack([
        sizes,
        sizes * np.log2(sizes + 1)
    ])

    model = LinearRegression()
    model.fit(X_feat, real_times)

    ml_pred = model.predict(X_feat)

    # --------------------------
    # plot (NO scaling)
    # --------------------------
    def normalize(curve, real):
      """
      shape 비교용 정규화 (scale 제거)
      """
      curve = np.array(curve)
      return curve / np.max(curve) * np.max(real)

      for name, curve in theory.items():
          plt.plot(
              sizes,
              normalize_to_real(curve, real_times),
              label=name
          )

    plt.figure(figsize=(12, 12))

    # =========================
    # 1. RAW SCALE (절대 시간)
    # =========================
    plt.subplot(3, 1, 1)

    plt.scatter(sizes, real_times, color="black", label="Real Data")

    plt.plot(sizes, theory["O(n)"], label="O(n)")
    plt.plot(sizes, theory["O(n log n)"], label="O(n log n)")
    plt.plot(sizes, ml_pred, "--", label="ML Fit")

    plt.title("1. Raw Scale (Absolute Time)")
    plt.ylabel("ms")
    plt.grid()
    plt.legend()

    # =========================
    # 2. NORMALIZED SCALE (shape 비교)
    # =========================
    plt.subplot(3, 1, 2)

    plt.scatter(sizes, real_times, color="black", label="Real Data")

    plt.plot(sizes, normalize(theory["O(n)"], real_times), label="O(n)")
    plt.plot(sizes, normalize(theory["O(n log n)"], real_times), label="O(n log n)")
    plt.plot(sizes, normalize(ml_pred, real_times), "--", label="ML Fit")

    plt.title("2. Normalized Scale (Shape Comparison)")
    plt.ylabel("scaled")
    plt.grid()
    plt.legend()

    # =========================
    # 3. LOG SCALE (growth rate)
    # =========================
    plt.subplot(3, 1, 3)

    plt.scatter(sizes, real_times, color="black", label="Real Data")

    plt.plot(sizes, theory["O(n)"], label="O(n)")
    plt.plot(sizes, theory["O(n log n)"], label="O(n log n)")
    plt.plot(sizes, ml_pred, "--", label="ML Fit")

    plt.yscale("log")

    plt.title("3. Log Scale (Growth Rate)")
    plt.xlabel("Input size (n)")
    plt.ylabel("log(ms)")
    plt.grid()
    plt.legend()

    plt.tight_layout()
    plt.show()

    print("real_times:", real_times)

In [ ]:
def mode_B(mode, size):

    arr = generate(size, mode)

    results = {}

    for name, f in algos.items():

        start = time.perf_counter()
        f(arr)
        results[name] = (time.perf_counter()-start)*1000

    plt.figure(figsize=(8,5))

    plt.bar(results.keys(), results.values())

    plt.title(f"[MODE B] Algorithm Comparison (n={size})")
    plt.ylabel("Time (ms)")
    plt.grid(axis="y")

    plt.show()

In [ ]:
mode_select = widgets.ToggleButtons(
    options=["Mode A", "Mode B"]
)

algo_select = widgets.Dropdown(
    options=list(algos.keys()),
    value="Kadane"
)

data_mode = widgets.Dropdown(
    options=["random", "monotonic", "alternating", "sparse"],
    value="random"
)

size_slider = widgets.IntSlider(
    value=400,
    min=100,
    max=1500,
    step=1
)

out = widgets.Output()

In [ ]:
def update(change):

    with out:
        clear_output(wait=True)

        if mode_select.value == "Mode A":
            mode_A(
                algo_select.value,
                data_mode.value,
                size_slider.value
            )

        else:
            mode_B(
                data_mode.value,
                size_slider.value
            )

In [ ]:
for w in [mode_select, algo_select, data_mode, size_slider]:
    w.observe(update, names="value")

display(mode_select, algo_select, data_mode, size_slider, out)

update(None)

ToggleButtons(options=('Mode A', 'Mode B'), value='Mode A')

Dropdown(index=1, options=('DnC', 'Kadane', 'Prefix'), value='Kadane')

Dropdown(options=('random', 'monotonic', 'alternating', 'sparse'), value='random')

IntSlider(value=1497, max=1500, min=100)

Output()

# LIS

In [ ]:
from bisect import bisect_left


def minimum_move_sort(arr):
    """
    최소 이동 정렬 분석 함수

    목표:
    - 배열에서 LIS(Longest Increasing Subsequence)를 찾는다.
    - LIS에 포함되지 않은 원소만 이동하면
      전체 배열을 정렬할 수 있다.

    핵심 원리:
    최소 이동 횟수 = 전체 길이 - LIS 길이

    시간복잡도:
    O(n log n)
    """

    # 배열 길이
    n = len(arr)

    # -----------------------------
    # tails:
    # tails[i] =
    # 길이가 i+1인 증가 부분수열 중
    # "가장 작은 마지막 값"
    #
    # 예:
    # tails = [1, 3, 7]
    #
    # 의미:
    # 길이 1 증가수열 마지막 최소값 = 1
    # 길이 2 증가수열 마지막 최소값 = 3
    # 길이 3 증가수열 마지막 최소값 = 7
    # -----------------------------
    tails = []

    # -----------------------------
    # tails_idx:
    # tails 값이 실제 배열의 어느 인덱스인지 저장
    #
    # LIS 복원을 위해 필요
    # -----------------------------
    tails_idx = []

    # -----------------------------
    # parent[i]:
    # arr[i] 이전에 연결되는 LIS 원소 위치
    #
    # LIS를 역추적하기 위해 사용
    # -----------------------------
    parent = [-1] * n

    # =============================
    # LIS 계산 시작
    # =============================
    for i, x in enumerate(arr):

        # ---------------------------------
        # x가 들어갈 위치를 binary search
        #
        # 예:
        # tails = [1,2,5]
        # x = 4
        #
        # 들어갈 위치:
        # pos = 2
        #
        # 결과:
        # tails = [1,2,4]
        # ---------------------------------
        pos = bisect_left(tails, x)

        # ---------------------------------
        # pos가 tails 끝이면
        # 새로운 LIS 길이 증가
        # ---------------------------------
        if pos == len(tails):
            tails.append(x)
            tails_idx.append(i)

        # ---------------------------------
        # 기존 값을 더 작은 값으로 교체
        #
        # 더 작은 끝값이 이후 확장에 유리
        # ---------------------------------
        else:
            tails[pos] = x
            tails_idx[pos] = i

        # ---------------------------------
        # parent 연결
        #
        # pos > 0 이면
        # 현재 원소 앞에는
        # 길이 pos인 LIS가 존재
        #
        # 그 마지막 원소를 연결
        # ---------------------------------
        if pos > 0:
            parent[i] = tails_idx[pos - 1]

    # =============================
    # LIS 복원
    # =============================

    lis = []

    # LIS 마지막 원소 인덱스
    cur = tails_idx[-1]

    # parent를 따라 역추적
    while cur != -1:
        lis.append(arr[cur])
        cur = parent[cur]

    # 뒤집어서 정상 순서로
    lis.reverse()

    # 최소 이동 횟수
    moves = n - len(lis)

    return lis, moves


# ==========================================
# 실행 코드
# ==========================================

arr = [3, 1, 2, 5, 4]

print("원본 배열:")
print(arr)

lis, moves = minimum_move_sort(arr)

print("\nLIS (이동하지 않아도 되는 원소들):")
print(lis)

print("\n최소 이동 횟수:")
print(moves)